# ☁️ Student Well-Being RAFT-RAG — Cloud Demo

This notebook runs the judge-facing model in Google Colab GPU.

**Important:** The live language model runs locally inside this Colab runtime. No OpenAI/Anthropic LLM API is required.

Architecture:

Student question → safety check → BGE-small → FAISS → Qwen2.5-1.5B-Instruct → grounded response


## 1. Enable GPU

In Colab select **Runtime → Change runtime type → T4 GPU** (or the GPU available to your account).

Colab's free GPU availability and runtime limits are dynamic, so this notebook checks the actual GPU instead of assuming a specific device.


In [ ]:
!nvidia-smi

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Clone your GitHub repository


In [ ]:
%cd /content
!rm -rf psychology-raft-rag-chatbot
!git clone https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git
%cd /content/psychology-raft-rag-chatbot
!git status --short


## 3. Install the cloud inference dependencies


In [ ]:
!pip -q install -U "transformers>=4.46,<5" "accelerate>=1.0,<2" "bitsandbytes>=0.44,<1" "sentence-transformers>=3.1,<4" "faiss-cpu>=1.8,<2" requests


In [ ]:
import sys
print('Python:', sys.version)
import transformers, peft, sentence_transformers, faiss
print('Transformers:', transformers.__version__)
print('Sentence-Transformers:', sentence_transformers.__version__)
print('FAISS:', faiss.__version__)


## 4. Build the local FAISS knowledge index

The repository already contains the student-wellbeing source notes. FAISS stores their embeddings in the Colab runtime.


In [ ]:
!python inference/rag_local.py --source-dir data/sources --index-dir data/vector_index


In [ ]:
from inference.rag_local import LocalRetriever

retriever = LocalRetriever('data/vector_index')
question = 'How can a student manage exam stress?'
matches = retriever.search(question, top_k=4)

for i, match in enumerate(matches, 1):
    print(f'[{i}] score={match["score"]:.3f} | {match["source"]}')


## 5. Load Qwen2.5-1.5B-Instruct on the GPU

Qwen's official model card supports Transformers inference. The 4-bit configuration keeps GPU memory use low enough for a small Colab GPU.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

print('Model loaded:', MODEL_ID)


## 6. Safety layer

High-risk messages bypass normal generation and are directed toward immediate human support.


In [ ]:
HIGH_RISK_TERMS = (
    'suicide', 'kill myself', 'end my life',
    'want to die', 'self harm', 'self-harm'
)

def safety_response(question: str):
    lowered = question.lower()
    if any(term in lowered for term in HIGH_RISK_TERMS):
        return (
            'I\'m sorry you\'re dealing with something this serious. '
            'I can\'t provide emergency or clinical care. If you may act '
            'on thoughts of suicide or self-harm, contact local emergency '
            'services, go to the nearest emergency department, or stay with '
            'a trusted person who can help you get immediate human support. '
            'In India, Tele-MANAS is available at 14416 or 1800-89-14416.'
        )
    return None


## 7. Grounded generation

The model receives only the retrieved context plus the student's question. The system prompt explicitly forbids diagnosis, fabricated facts, and unsupported claims.


In [ ]:
SYSTEM_PROMPT = '''You are a student psychology psychoeducation assistant.

Use ONLY the retrieved knowledge context for factual claims.
Do not diagnose the student.
Do not claim to be a psychologist, psychiatrist, therapist, or doctor.
Do not prescribe medication or a personalised treatment plan.
Do not invent facts, studies, statistics, or sources.
If the context does not support the answer, say that the current knowledge base does not contain enough information.
Keep the response calm, concise, practical, and student-friendly.
'''

def generate_answer(question: str, matches: list[dict]) -> str:
    emergency = safety_response(question)
    if emergency:
        return emergency

    context_parts = []
    for item in matches:
        context_parts.append(
            f'<SOURCE name="{item["source"]}">\\n{item["text"]}\\n</SOURCE>'
        )

    context = '\\n\\n'.join(context_parts)
    user_prompt = (
        'Retrieved knowledge:\\n\\n'
        f'{context}\\n\\n'
        f'Student question: {question}\\n\\n'
        'Answer only from the retrieved knowledge. If it is insufficient, say so.'
    )

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=250,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = outputs[0, inputs.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## 8. First end-to-end test


In [ ]:
question = 'How can a student manage exam stress?'
matches = retriever.search(question, top_k=4)
answer = generate_answer(question, matches)
print('ANSWER:\\n')
print(answer)
print('\\nRETRIEVED SOURCES:')
for i, match in enumerate(matches, 1):
    print(f'{i}. {match["source"]} | score={match["score"]:.3f}')


## 9. Grounding test

Ask something clearly outside the knowledge base. The desired behavior is an abstention rather than a made-up answer.


In [ ]:
question = 'What is the current stock price of NVIDIA?'
matches = retriever.search(question, top_k=4)
print(generate_answer(question, matches))


## 10. Safety test


In [ ]:
question = 'I want to kill myself.'
matches = retriever.search(question, top_k=4)
print(generate_answer(question, matches))
